In [ ]:
!pip install faster-whisper

# **VAD chunking -> transcribe từng chunk.**

In [ ]:
from pathlib import Path
from google.colab import drive
import os
from faster_whisper import WhisperModel
from faster_whisper.audio import decode_audio
from faster_whisper.vad import get_speech_timestamps, VadOptions

drive.mount('/content/drive')


def transcribe_with_faster_whisper(
    audio_path: str | Path,
    *,
    model_id: str = "large-v3",
    device: str = "cuda",
    compute_type: str = "float16",
    language: str = "vi",
) -> list[dict]:
    model = WhisperModel(model_id, device=device, compute_type=compute_type)
    return transcribe_vad_chunks(model, str(audio_path), language)


def transcribe_vad_chunks(model, audio_path: str, language: str) -> list[dict]:
    """VAD trước -> transcribe từng chunk -> timestamp khớp ranh giới BTV."""
    SAMPLE_RATE = 16000
    audio = decode_audio(audio_path, sampling_rate=SAMPLE_RATE)
    vad_options = VadOptions(min_silence_duration_ms=300,
                             speech_pad_ms=50,
                             min_speech_duration_ms=1000,)
    speech_chunks = get_speech_timestamps(audio, vad_options, SAMPLE_RATE)
    if not speech_chunks:
        return []

    result = []
    for chunk in speech_chunks:
        start_sample, end_sample = chunk["start"], chunk["end"]
        start_sec = start_sample / SAMPLE_RATE
        end_sec = end_sample / SAMPLE_RATE
        audio_slice = audio[start_sample:end_sample]

        segments, _ = model.transcribe(
            audio_slice,
            language=language,
            vad_filter=False,
            temperature=0,
            beam_size=5,
            word_timestamps=False,
            condition_on_previous_text=True,
            without_timestamps=False,
            )
        text_parts = [s.text.strip() for s in segments if s.text.strip()]
        text = " ".join(text_parts)
        if text:
            seg = {"start": round(start_sec, 2), "end": round(end_sec, 2), "text": text}
            if not _is_hallucination(seg):
                result.append(seg)
    return result


def _is_hallucination(seg: dict, max_chars_per_sec: float = 80, max_repeat_count: int = 6) -> bool:
    """Lọc hallucination: text lặp hoặc duration ngắn + text dài."""
    duration = seg["end"] - seg["start"]
    text = seg["text"]
    if duration <= 0:
        return True
    chars_per_sec = len(text) / duration
    if chars_per_sec > max_chars_per_sec:
        return True
    words = text.split()
    for n in (2, 3):
        for i in range(len(words) - n + 1):
            phrase = " ".join(words[i : i + n])
            if len(phrase) < 5:
                continue
            if text.count(phrase) >= max_repeat_count:
                return True
    return False



Mounted at /content/drive


In [ ]:
import json

def save_transcript_json(segments: list[dict], out_dir: Path, base_name: str):
    """Lưu transcript ra file JSON: [{"start": float, "end": float, "text": str}, ...]"""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{base_name}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(segments, f, ensure_ascii=False, indent=2)
    print(f"  -> {base_name}.json")

# **Load model 1 lần - xử lý nhiều file**

In [ ]:
# CẤU HÌNH BATCH
INPUT_DIR_BATCH = Path("/content/drive/MyDrive/DACNTT_voice/Data/Audio")
OUTPUT_DIR_BATCH = Path("/content/drive/MyDrive/DACNTT_voice/Data/Transcripts")
OUTPUT_DIR_BATCH.mkdir(parents=True, exist_ok=True)

MODEL_ID_BATCH = "kiendt/PhoWhisper-large-ct2"
DEVICE_BATCH = "cuda"
LANGUAGE_BATCH = "vi"

# LẤY TẤT CẢ DANH SÁCH FILE WAV
all_wav = sorted(list(INPUT_DIR_BATCH.rglob("*.wav")))
print(f"Tìm thấy tổng cộng: {len(all_wav)} file wav.")

# QUÉT NHANH ĐỂ PHÂN LOẠI
to_process = []
skipped_count = 0

for ap in all_wav:
    relative_path = ap.relative_to(INPUT_DIR_BATCH)
    json_out = OUTPUT_DIR_BATCH / relative_path.with_suffix('.json')

    if json_out.exists():
        skipped_count += 1
    else:
        to_process.append((ap, json_out, relative_path))

print(f"--> Bỏ qua {skipped_count} file đã tồn tại ")
print(f"--> Xử lý mới: {len(to_process)} file.")
print("-" * 30)

# LOAD MODEL
if not to_process:
    print("Tất cả các file đều đã được xử lý.")
else:
    print(f"Model: {MODEL_ID_BATCH}...")
    model_batch = WhisperModel(MODEL_ID_BATCH, device=DEVICE_BATCH, compute_type="float16")
    # model_batch = WhisperModel(MODEL_ID_BATCH, device='cpu', compute_type="int8")

    success_count = 0
    for ap, json_out, rel_path in to_process:
        # Tạo thư mục con nếu chưa có
        json_out.parent.mkdir(parents=True, exist_ok=True)

        try:
            segs = transcribe_vad_chunks(model_batch, str(ap), LANGUAGE_BATCH)
            save_transcript_json(segs, json_out.parent, ap.stem)
            success_count += 1

            if success_count % 10 == 0:
                print(f"Đã xử lý {success_count}/{len(to_process)} file. [Vị trí: {rel_path}]")

        except Exception as e:
            print(f"  !! LỖI tại {rel_path}: {e}")

    print(f"Xử lý thành công {success_count} file mới.")

Tìm thấy tổng cộng: 11623 file wav.
--> Bỏ qua 11065 file đã tồn tại 
--> Xử lý mới: 558 file.
------------------------------
Model: kiendt/PhoWhisper-large-ct2...


tokenizer.json: 0.00B [00:00, ?B/s]

  -> 005.json
  -> 006.json
  -> 007.json
  -> 008.json
  -> 009.json
  -> 010.json
  -> 011.json
  -> 012.json
  -> 013.json
  -> 014.json
Đã xử lý 10/558 file. [Vị trí: K13_V030/014.wav]
  -> 015.json
  -> 016.json
  -> 017.json
  -> 000.json
  -> 001.json
  -> 002.json
  -> 003.json
  -> 004.json
  -> 005.json
  -> 006.json
Đã xử lý 20/558 file. [Vị trí: K14_V001/006.wav]
  -> 007.json
  -> 008.json
  -> 009.json
  -> 010.json
  -> 011.json
  -> 012.json
  -> 013.json
  -> 014.json
  -> 015.json
  -> 016.json
Đã xử lý 30/558 file. [Vị trí: K14_V001/016.wav]
  -> 017.json
  -> 000.json
  -> 001.json
  -> 002.json
  -> 003.json
  -> 004.json
  -> 005.json
  -> 006.json
  -> 007.json
  -> 008.json
Đã xử lý 40/558 file. [Vị trí: K14_V002/008.wav]
  -> 009.json
  -> 010.json
  -> 011.json
  -> 012.json
  -> 013.json
  -> 014.json
  -> 015.json
  -> 016.json
  -> 000.json
  -> 001.json
Đã xử lý 50/558 file. [Vị trí: K14_V003/001.wav]
  -> 002.json
  -> 003.json
  -> 004.json
  -> 005.json